# EIA Weekly U.S. Petroleum Stocks Exploration

Weekly ending stocks (thousand barrels) from the U.S. Energy Information Administration:
- **Crude Oil** — total commercial stocks (WCRSTUS1, from 1982)
- **Crude Oil in SPR** — Strategic Petroleum Reserve (WCSSTUS1, from 1982)
- **Distillate Fuel Oil** — heating oil + diesel (WDISTUS1, from 1982)
- **Total Gasoline** — finished motor gasoline (WGTSTUS1, from 1990)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True

In [2]:
DATA_DIR = "data"

# EIA .xls files share the same layout: real data starts on row 2 of sheet "Data 1"
_EIA_SKIPROWS = 2

def load_eia(filename, col_name):
    df = pd.read_excel(
        f"{DATA_DIR}/{filename}",
        sheet_name="Data 1",
        skiprows=_EIA_SKIPROWS,
        index_col=0,
        parse_dates=True,
    )
    df.index.name = "date"
    df.columns = [col_name]
    return df

crude      = load_eia("WCRSTUS1w.xls", "crude_oil")
spr        = load_eia("WCSSTUS1w.xls", "spr")
distillate = load_eia("WDISTUS1w.xls", "distillate")
gasoline   = load_eia("WGTSTUS1w.xls", "gasoline")

# Combined view (inner join aligns on common dates; gasoline starts 1990)
stocks = crude.join([spr, distillate, gasoline], how="outer")
stocks.index = pd.to_datetime(stocks.index)
stocks = stocks.sort_index()

print(f"Date range : {stocks.index.min().date()} → {stocks.index.max().date()}")
print(f"Rows       : {len(stocks):,}")
stocks.tail()

Date range : 1982-08-20 → 2026-06-12
Rows       : 2,281


,crude_oil,spr,distillate,gasoline
date,,,,
2026-05-15,819188,374175,102906,214163.0
2026-05-22,806798,365112,100799,211591.0
2026-05-29,790831,357119,102301,214955.0
2026-06-05,775677,349192,102101,215141.0
2026-06-12,758473,340251,103052,214235.0


In [3]:
import plotly.graph_objects as go
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────────
CURRENT_YEAR = int(stocks.index.year.max())
OUTPUT_FILE  = "petroleum_seasonality_v1.html"   # versioned output

PRODUCTS = {
    "crude_oil":  ("Crude Oil",              "WCRSTUS1", "1982"),
    "gasoline":   ("Total Gasoline",         "WGTSTUS1", "1990"),
    "distillate": ("Distillate (Diesel)",    "WDISTUS1", "1982"),
    "spr":        ("Strategic Reserve (SPR)","WCSSTUS1", "1982"),
}

# week = (day-of-year - 1) // 7 + 1, capped at 52 (matches EIA convention)
doy = stocks.index.day_of_year.to_series(index=stocks.index)
stocks["year"] = stocks.index.year
stocks["week"] = ((doy - 1) // 7 + 1).clip(upper=52).astype(int)

# ── Decade color palettes ─────────────────────────────────────────────────────
DECADE_PALETTES = {
    1980: ((168, 168, 175, 0.28), ( 85,  85, 102, 0.84)),  # gray
    1990: ((120, 200, 135, 0.28), ( 18, 122,  42, 0.86)),  # green
    2000: ((118, 168, 228, 0.28), ( 10,  72, 172, 0.88)),  # blue
    2010: ((248, 178,  82, 0.28), (185,  85,   5, 0.88)),  # amber / orange
    2020: ((200, 148, 228, 0.28), (108,  35, 162, 0.88)),  # purple
}
DECADE_LEGEND = {
    1980: ("1980s", "rgba(125, 125, 138, 0.82)"),
    1990: ("1990s", "rgba( 62, 158,  86, 0.86)"),
    2000: ("2000s", "rgba( 55, 118, 198, 0.88)"),
    2010: ("2010s", "rgba(215, 130,  42, 0.88)"),
    2020: ("2020s", "rgba(152,  88, 195, 0.88)"),
}

def decade_rgba(year, frac):
    d = (year // 10) * 10
    (lr, lg, lb, la), (dr, dg, db, da) = DECADE_PALETTES[d]
    r = int(lr + frac * (dr - lr))
    g = int(lg + frac * (dg - lg))
    b = int(lb + frac * (db - lb))
    a = round(la + frac * (da - la), 2)
    return f"rgba({r},{g},{b},{a})"

# ── Build traces ──────────────────────────────────────────────────────────────
fig = go.Figure()
product_trace_ranges = {}

for col, (label, series_id, start_yr) in PRODUCTS.items():
    s = stocks[["year", "week", col]].dropna(subset=[col])
    years       = sorted(s["year"].unique())
    prior_years = [y for y in years if y < CURRENT_YEAR]
    first_idx   = len(fig.data)

    decade_groups = defaultdict(list)
    for y in prior_years:
        decade_groups[(y // 10) * 10].append(y)

    for yr in prior_years:
        d    = (yr // 10) * 10
        grp  = decade_groups[d]
        frac = grp.index(yr) / max(len(grp) - 1, 1)
        yd   = s[s["year"] == yr].sort_values("week")
        fig.add_trace(go.Scatter(
            x=yd["week"], y=yd[col],
            mode="lines",
            line=dict(color=decade_rgba(yr, frac), width=1),
            showlegend=False,
            visible=(col == "crude_oil"),
            hovertemplate=(
                f"Year: {yr}<br>Week: %{{x}}<br>"
                "%{y:,.0f} thousand barrels<extra></extra>"
            ),
        ))

    if CURRENT_YEAR in years:
        yd = s[s["year"] == CURRENT_YEAR].sort_values("week")
        fig.add_trace(go.Scatter(
            x=yd["week"], y=yd[col],
            mode="lines",
            line=dict(color="#c0392b", width=2.8),
            showlegend=False,
            visible=(col == "crude_oil"),
            hovertemplate=(
                f"Year: {CURRENT_YEAR}<br>Week: %{{x}}<br>"
                "%{y:,.0f} thousand barrels<extra></extra>"
            ),
        ))

    product_trace_ranges[col] = (first_idx, len(fig.data) - 1)

legend_start = len(fig.data)
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="lines",
    line=dict(color="#c0392b", width=2.8),
    name=str(CURRENT_YEAR), showlegend=True,
))
for decade in sorted(DECADE_LEGEND, reverse=True):
    lbl, clr = DECADE_LEGEND[decade]
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="lines",
        line=dict(color=clr, width=1.8),
        name=lbl, showlegend=True,
    ))
legend_idxs = list(range(legend_start, len(fig.data)))

def source_note(sid, sy):
    return (
        f"Source: EIA series {sid} — Weekly U.S. Ending Stocks, "
        f"{sy}–present, thousand barrels.  "
        f"Latest year on chart: {CURRENT_YEAR}.  "
        "Week = (day-of-year ÷ 7), capped at 52."
    )

n_total = len(fig.data)
buttons = []
for col, (label, series_id, start_yr) in PRODUCTS.items():
    lo, hi  = product_trace_ranges[col]
    visible = [False] * n_total
    for i in range(lo, hi + 1):
        visible[i] = True
    for i in legend_idxs:
        visible[i] = True
    buttons.append(dict(
        label=label, method="update",
        args=[
            {"visible": visible},
            {"annotations[2].text": source_note(series_id, start_yr)},
        ],
    ))

fig.update_layout(
    font=dict(family="Arial, Helvetica, sans-serif"),
    plot_bgcolor="white", paper_bgcolor="white",
    title=dict(
        text="U.S. Petroleum Inventories — Weekly Seasonality",
        font=dict(size=22, color="#111"),
        x=0.01, xanchor="left", y=0.97, yanchor="top",
    ),
    xaxis=dict(
        title="Week of year",
        tickmode="array", tickvals=list(range(1, 53, 2)),
        range=[0.5, 52.5],
        gridcolor="rgba(210,215,225,0.7)", showgrid=True, zeroline=False,
    ),
    yaxis=dict(
        title="Thousand barrels",
        tickformat=",", separatethousands=True,
        gridcolor="rgba(210,215,225,0.7)", showgrid=True, zeroline=False,
    ),
    legend=dict(
        x=0.01, y=0.885, xanchor="left", yanchor="top",
        orientation="h", bgcolor="rgba(0,0,0,0)", font=dict(size=12),
    ),
    updatemenus=[dict(
        buttons=buttons, direction="down", showactive=True,
        x=0.085, xanchor="left", y=1.09, yanchor="top",
        bgcolor="white", bordercolor="#bbb", font=dict(size=13),
    )],
    annotations=[
        dict(text="Product:", x=0.01, y=1.095,
             xref="paper", yref="paper", showarrow=False,
             font=dict(size=13, color="#333"), xanchor="left"),
        dict(
            text="Stocks by week of year (1–52), one line per year. Latest year bolded in red.",
            x=0.01, y=1.045, xref="paper", yref="paper",
            showarrow=False, font=dict(size=12, color="#666"), xanchor="left",
        ),
        dict(text=source_note("WCRSTUS1", "1982"),
             x=0.01, y=-0.10, xref="paper", yref="paper",
             showarrow=False, font=dict(size=10, color="#888"), xanchor="left"),
    ],
    margin=dict(t=130, b=90, l=80, r=40),
    width=1100, height=620,
)

fig.write_html(OUTPUT_FILE, include_plotlyjs="cdn")
print(f"Saved: {OUTPUT_FILE}")

Saved: petroleum_seasonality_v1.html


In [4]:
"""
v2: Replaces Plotly's built-in chrome with the D4TP design system from heatmap_v22:
  - CSS custom properties (light + dark mode), system font stack
  - Native <select> in a styled .ctrls panel
  - Custom HTML legend with decade color swatches
  - .notes source attribution line
  - D4TP credit bar with logo
  - Plotly renders only the chart lines via Plotly.react() on product change
"""
import re, json, os

# ── Serialise data for JS ─────────────────────────────────────────────────────
chart_data = {}
for col in PRODUCTS:
    s = stocks[["year", "week", col]].dropna(subset=[col]).copy()
    s["decade"] = (s["year"] // 10) * 10
    rows = []
    for yr, grp in s.groupby("year"):
        g = grp.sort_values("week")
        rows.append({
            "year":   int(yr),
            "decade": int(g["decade"].iloc[0]),
            "x":      g["week"].tolist(),
            "y":      g[col].tolist(),
        })
    chart_data[col] = rows

# ── Source notes per product ──────────────────────────────────────────────────
def src(sid, sy):
    return (f"EIA series {sid} — Weekly U.S. Ending Stocks, {sy}–present, "
            f"thousand barrels. Latest year: {CURRENT_YEAR}. "
            "Week = (day-of-year / 7), capped at 52.")

source_notes = {col: src(sid, sy) for col, (_, sid, sy) in PRODUCTS.items()}

# ── Pull D4TP credit bar from the reference heatmap ──────────────────────────
HEATMAP_PATH = r"C:\Users\amand\Downloads\heatmap_v22.html"
credit_bar_html = ""
if os.path.exists(HEATMAP_PATH):
    with open(HEATMAP_PATH, encoding="utf-8") as f:
        raw = f.read()
    m = re.search(r'<div class="credit-bar">(.*?)</div>\s*\n', raw, re.DOTALL)
    if m:
        credit_bar_html = m.group(1).strip()

# ── Legend HTML ───────────────────────────────────────────────────────────────
DECADE_MID = {
    1980: "rgba(125,125,138,0.82)",
    1990: "rgba(62,158,86,0.86)",
    2000: "rgba(55,118,198,0.88)",
    2010: "rgba(215,130,42,0.88)",
    2020: "rgba(152,88,195,0.88)",
}
legend_items = (
    f'  <div class="legend-item">'
    f'<span class="leg-line" style="background:#c0392b;height:3px;opacity:1"></span>'
    f'<span class="leg-label">{CURRENT_YEAR}</span></div>'
)
for dec in sorted(DECADE_MID, reverse=True):
    label = f"{dec}s"
    legend_items += (
        f'\n  <div class="legend-item">'
        f'<span class="leg-line" style="background:{DECADE_MID[dec]}"></span>'
        f'<span class="leg-label">{label}</span></div>'
    )

select_options = "\n".join(
    f'      <option value="{col}">{label}</option>'
    for col, (label, _, _) in PRODUCTS.items()
)

OUTPUT_V2 = "petroleum_seasonality_v2.html"

HTML = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>U.S. Petroleum Inventories - Weekly Seasonality</title>
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
:root {{
  --bg-primary:   #ffffff;  --bg-secondary: #f5f4ef;  --bg-tertiary: #efece4;
  --text-primary: #1a1a1a;  --text-secondary: #555550; --text-tertiary: #888880;
  --border: rgba(0,0,0,0.12); --border-strong: rgba(0,0,0,0.25);
}}
@media (prefers-color-scheme: dark) {{
  :root {{
    --bg-primary: #1a1a1a;   --bg-secondary: #232220;  --bg-tertiary: #2c2c2a;
    --text-primary: #e8e6df; --text-secondary: #a8a59b; --text-tertiary: #6b6962;
    --border: rgba(255,255,255,0.12); --border-strong: rgba(255,255,255,0.25);
  }}
}}
*{{box-sizing:border-box;margin:0;padding:0;}}
html,body{{background:var(--bg-primary);}}
body{{font-family:-apple-system,BlinkMacSystemFont,"Inter","Segoe UI",sans-serif;
      color:var(--text-primary);padding:20px;max-width:1100px;margin:0 auto;line-height:1.5;}}
.hdr{{margin-bottom:14px;}}
.hdr h1{{font-size:22px;font-weight:600;margin-bottom:4px;line-height:1.25;}}
.hdr p{{font-size:13px;color:var(--text-secondary);max-width:700px;}}
.ctrls{{display:flex;align-items:center;gap:12px;margin-bottom:12px;padding:12px 14px;
        background:var(--bg-secondary);border-radius:10px;}}
.ctrl-grp{{display:flex;flex-direction:column;gap:5px;}}
.ctrl-grp label{{font-size:11px;color:var(--text-secondary);
                 text-transform:uppercase;letter-spacing:0.04em;font-weight:600;}}
.ctrl-grp select{{font-size:14px;padding:8px 32px 8px 11px;
  background:var(--bg-primary);color:var(--text-primary);
  border:0.5px solid var(--border);border-radius:7px;font-family:inherit;cursor:pointer;
  -webkit-appearance:none;appearance:none;
  background-image:url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='10' height='6' viewBox='0 0 10 6'><path d='M0 0l5 6 5-6z' fill='%23555'/></svg>");
  background-repeat:no-repeat;background-position:right 12px center;min-height:38px;}}
.legend{{display:flex;flex-wrap:wrap;gap:6px 16px;margin-bottom:8px;
         font-size:12px;color:var(--text-secondary);}}
.legend-item{{display:flex;align-items:center;gap:7px;}}
.leg-line{{display:inline-block;width:22px;height:2px;border-radius:1px;flex-shrink:0;}}
.leg-label{{white-space:nowrap;}}
#chart{{width:100%;}}
.notes{{font-size:11px;color:var(--text-secondary);margin-top:12px;line-height:1.6;
        padding-top:12px;border-top:0.5px solid var(--border);}}
.notes strong{{color:var(--text-primary);font-weight:500;}}
.credit-bar{{display:flex;justify-content:flex-end;align-items:center;
             margin-top:14px;padding-top:12px;border-top:0.5px solid var(--border);}}
.d4tp-logo{{display:inline-block;text-decoration:none;opacity:0.85;transition:opacity 0.15s;}}
.d4tp-logo:hover{{opacity:1;}}
.d4tp-logo svg{{display:block;height:24px;width:auto;}}
.logo-light-mode{{display:inline-block;}} .logo-dark-mode{{display:none;}}
@media(prefers-color-scheme:dark){{
  .logo-light-mode{{display:none;}} .logo-dark-mode{{display:inline-block;}}
}}
@media(max-width:700px){{body{{padding:14px;}}.hdr h1{{font-size:18px;}}.ctrls{{flex-wrap:wrap;}}}}
</style>
</head>
<body>

<div class="hdr">
  <h1>U.S. Petroleum Inventories &#8212; Weekly Seasonality</h1>
  <p>Stocks by week of year (1&#8211;52), one line per year. {CURRENT_YEAR} bolded in red; prior years shaded by decade.</p>
</div>

<div class="ctrls">
  <div class="ctrl-grp">
    <label for="product-sel">Product</label>
    <select id="product-sel">
{select_options}
    </select>
  </div>
</div>

<div class="legend">
{legend_items}
</div>

<div id="chart"></div>

<div class="notes">
  <strong>Source:</strong> <span id="source-note">{source_notes['crude_oil']}</span>
</div>

<div class="credit-bar">
  {credit_bar_html}
</div>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js" charset="utf-8"></script>
<script>
const CURRENT_YEAR = {CURRENT_YEAR};
const CHART_DATA   = {json.dumps(chart_data)};
const SOURCE_NOTES = {json.dumps(source_notes)};

const DECADE_PALETTES = {{
  1980:{{lr:168,lg:168,lb:175,la:0.28,dr:85, dg:85, db:102,da:0.84}},
  1990:{{lr:120,lg:200,lb:135,la:0.28,dr:18, dg:122,db:42, da:0.86}},
  2000:{{lr:118,lg:168,lb:228,la:0.28,dr:10, dg:72, db:172,da:0.88}},
  2010:{{lr:248,lg:178,lb:82, la:0.28,dr:185,dg:85, db:5,  da:0.88}},
  2020:{{lr:200,lg:148,lb:228,la:0.28,dr:108,dg:35, db:162,da:0.88}},
}};

function decadeRgba(decade, frac) {{
  const p = DECADE_PALETTES[decade];
  const r = Math.round(p.lr + frac*(p.dr-p.lr));
  const g = Math.round(p.lg + frac*(p.dg-p.lg));
  const b = Math.round(p.lb + frac*(p.db-p.lb));
  const a = (p.la + frac*(p.da-p.la)).toFixed(2);
  return `rgba(${{r}},${{g}},${{b}},${{a}})`;
}}

function buildTraces(product) {{
  const series = CHART_DATA[product];
  const decadeGroups = {{}};
  series.forEach(s => {{
    if (s.year < CURRENT_YEAR) {{
      if (!decadeGroups[s.decade]) decadeGroups[s.decade] = [];
      decadeGroups[s.decade].push(s.year);
    }}
  }});
  return series.map(s => {{
    const isCur = s.year === CURRENT_YEAR;
    let color, width;
    if (isCur) {{ color='#c0392b'; width=2.8; }}
    else {{
      const grp=decadeGroups[s.decade];
      color=decadeRgba(s.decade, grp.indexOf(s.year)/Math.max(grp.length-1,1));
      width=1;
    }}
    return {{
      x:s.x, y:s.y, type:'scatter', mode:'lines',
      line:{{color,width}},
      hovertemplate:`Year: ${{s.year}}<br>Week: %{{x}}<br>%{{y:,.0f}} thousand barrels<extra></extra>`,
    }};
  }});
}}

const layout = {{
  margin:{{t:14,b:52,l:78,r:16}},
  plot_bgcolor:'rgba(0,0,0,0)', paper_bgcolor:'rgba(0,0,0,0)',
  xaxis:{{
    title:{{text:'Week of year',font:{{size:12}}}},
    tickmode:'array',
    tickvals:[1,3,5,7,9,11,13,15,17,19,21,23,25,27,29,31,33,35,37,39,41,43,45,47,49,51],
    range:[0.5,52.5], gridcolor:'rgba(210,215,225,0.7)', showgrid:true, zeroline:false,
  }},
  yaxis:{{
    title:{{text:'Thousand barrels',font:{{size:12}}}},
    tickformat:',', separatethousands:true,
    gridcolor:'rgba(210,215,225,0.7)', showgrid:true, zeroline:false,
  }},
  showlegend:false,
  font:{{family:'-apple-system,BlinkMacSystemFont,"Inter","Segoe UI",sans-serif',size:12}},
  height:500,
}};

const config = {{responsive:true, displayModeBar:false}};

Plotly.newPlot('chart', buildTraces('crude_oil'), layout, config);

document.getElementById('product-sel').addEventListener('change', function() {{
  Plotly.react('chart', buildTraces(this.value), layout, config);
  document.getElementById('source-note').textContent = SOURCE_NOTES[this.value];
}});
</script>
</body>
</html>"""

with open(OUTPUT_V2, "w", encoding="utf-8") as f:
    f.write(HTML)
print(f"Saved: {OUTPUT_V2}  ({os.path.getsize(OUTPUT_V2):,} bytes)")

Saved: petroleum_seasonality_v2.html  (159,849 bytes)


In [5]:
"""
v3: Adds decade toggle pills to the controls panel.
  - Each pill colored with its decade's color when active
  - All decades on by default; toggling hides/shows those lines
  - Current year always shown regardless of decade filter
  - Legend items dim when their decade is deselected
"""
import re, json, os

# ── Serialise data (same as v2) ───────────────────────────────────────────────
chart_data = {}
for col in PRODUCTS:
    s = stocks[["year", "week", col]].dropna(subset=[col]).copy()
    s["decade"] = (s["year"] // 10) * 10
    rows = []
    for yr, grp in s.groupby("year"):
        g = grp.sort_values("week")
        rows.append({
            "year":   int(yr),
            "decade": int(g["decade"].iloc[0]),
            "x":      g["week"].tolist(),
            "y":      g[col].tolist(),
        })
    chart_data[col] = rows

# Which decades actually appear across all products
all_decades = sorted({row["decade"] for series in chart_data.values() for row in series
                      if row["year"] < CURRENT_YEAR})

def src(sid, sy):
    return (f"EIA series {sid} — Weekly U.S. Ending Stocks, {sy}–present, "
            f"thousand barrels. Latest year: {CURRENT_YEAR}. "
            "Week = (day-of-year / 7), capped at 52.")

source_notes = {col: src(sid, sy) for col, (_, sid, sy) in PRODUCTS.items()}

# ── Pull credit bar from reference heatmap ────────────────────────────────────
HEATMAP_PATH = r"C:\Users\amand\Downloads\heatmap_v22.html"
credit_bar_html = ""
if os.path.exists(HEATMAP_PATH):
    with open(HEATMAP_PATH, encoding="utf-8") as f:
        raw = f.read()
    m = re.search(r'<div class="credit-bar">(.*?)</div>\s*\n', raw, re.DOTALL)
    if m:
        credit_bar_html = m.group(1).strip()

# ── Decade config (shared Python→HTML) ───────────────────────────────────────
DECADE_CFG = {
    1980: {"mid": "rgba(105,105,120,0.90)", "label": "1980s",
           "lr":168,"lg":168,"lb":175,"la":0.28, "dr":85, "dg":85, "db":102,"da":0.84},
    1990: {"mid": "rgba(18,122,42,0.90)",   "label": "1990s",
           "lr":120,"lg":200,"lb":135,"la":0.28, "dr":18, "dg":122,"db":42, "da":0.86},
    2000: {"mid": "rgba(10,72,172,0.90)",   "label": "2000s",
           "lr":118,"lg":168,"lb":228,"la":0.28, "dr":10, "dg":72, "db":172,"da":0.88},
    2010: {"mid": "rgba(185,85,5,0.90)",    "label": "2010s",
           "lr":248,"lg":178,"lb":82, "la":0.28, "dr":185,"dg":85, "db":5,  "da":0.88},
    2020: {"mid": "rgba(108,35,162,0.90)",  "label": "2020s",
           "lr":200,"lg":148,"lb":228,"la":0.28, "dr":108,"dg":35, "db":162,"da":0.88},
}

# Legend items: current year then decades newest→oldest
legend_items = (
    f'  <div class="legend-item">'
    f'<span class="leg-line" style="background:#c0392b;height:3px;opacity:1"></span>'
    f'<span class="leg-label">{CURRENT_YEAR}</span></div>'
)
for dec in sorted(DECADE_CFG, reverse=True):
    if dec not in all_decades:
        continue
    d = DECADE_CFG[dec]
    legend_items += (
        f'\n  <div class="legend-item" data-decade="{dec}">'
        f'<span class="leg-line" style="background:{d["mid"]}"></span>'
        f'<span class="leg-label">{d["label"]}</span></div>'
    )

# Decade toggle pills HTML
decade_pills_html = ""
for dec in all_decades:
    d = DECADE_CFG[dec]
    decade_pills_html += (
        f'<button class="decade-pill active" data-decade="{dec}" '
        f'data-color="{d["mid"]}">{d["label"]}</button>'
    )

select_options = "\n".join(
    f'      <option value="{col}">{label}</option>'
    for col, (label, _, _) in PRODUCTS.items()
)

OUTPUT_V3 = "petroleum_seasonality_v3.html"

HTML = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>U.S. Petroleum Inventories - Weekly Seasonality</title>
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
:root {{
  --bg-primary:    #ffffff; --bg-secondary: #f5f4ef; --bg-tertiary: #efece4;
  --text-primary:  #1a1a1a; --text-secondary: #555550; --text-tertiary: #888880;
  --border: rgba(0,0,0,0.12); --border-strong: rgba(0,0,0,0.25);
}}
@media (prefers-color-scheme: dark) {{
  :root {{
    --bg-primary: #1a1a1a;   --bg-secondary: #232220; --bg-tertiary: #2c2c2a;
    --text-primary: #e8e6df; --text-secondary: #a8a59b; --text-tertiary: #6b6962;
    --border: rgba(255,255,255,0.12); --border-strong: rgba(255,255,255,0.25);
  }}
}}
*{{box-sizing:border-box;margin:0;padding:0;}}
html,body{{background:var(--bg-primary);}}
body{{font-family:-apple-system,BlinkMacSystemFont,"Inter","Segoe UI",sans-serif;
      color:var(--text-primary);padding:20px;max-width:1100px;margin:0 auto;line-height:1.5;}}
.hdr{{margin-bottom:14px;}}
.hdr h1{{font-size:22px;font-weight:600;margin-bottom:4px;line-height:1.25;}}
.hdr p{{font-size:13px;color:var(--text-secondary);max-width:700px;}}

/* Controls */
.ctrls{{
  display:grid;
  grid-template-columns:auto 1fr;
  gap:14px;
  margin-bottom:12px;padding:12px 14px;
  background:var(--bg-secondary);border-radius:10px;
  align-items:end;
}}
.ctrl-grp{{display:flex;flex-direction:column;gap:5px;}}
.ctrl-grp label{{
  font-size:11px;color:var(--text-secondary);
  text-transform:uppercase;letter-spacing:0.04em;font-weight:600;
}}
.ctrl-grp select{{
  font-size:14px;padding:8px 32px 8px 11px;
  background:var(--bg-primary);color:var(--text-primary);
  border:0.5px solid var(--border);border-radius:7px;font-family:inherit;cursor:pointer;
  -webkit-appearance:none;appearance:none;
  background-image:url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='10' height='6' viewBox='0 0 10 6'><path d='M0 0l5 6 5-6z' fill='%23555'/></svg>");
  background-repeat:no-repeat;background-position:right 12px center;min-height:38px;
}}

/* Decade pills */
.decade-pills{{display:flex;gap:5px;flex-wrap:wrap;}}
.decade-pill{{
  padding:8px 14px;font-size:13px;
  background:var(--bg-primary);color:var(--text-secondary);
  border:0.5px solid var(--border);border-radius:7px;
  cursor:pointer;font-family:inherit;
  transition:background 0.12s,color 0.12s,border-color 0.12s,opacity 0.12s;
  min-height:38px;white-space:nowrap;
}}
.decade-pill:hover{{border-color:var(--border-strong);}}
.decade-pill.off{{
  background:var(--bg-primary) !important;
  color:var(--text-tertiary) !important;
  border-color:var(--border) !important;
  opacity:0.55;
}}

/* Legend */
.legend{{display:flex;flex-wrap:wrap;gap:6px 16px;margin-bottom:8px;
         font-size:12px;color:var(--text-secondary);}}
.legend-item{{display:flex;align-items:center;gap:7px;transition:opacity 0.15s;}}
.leg-line{{display:inline-block;width:22px;height:2px;border-radius:1px;flex-shrink:0;}}
.leg-label{{white-space:nowrap;}}

#chart{{width:100%;}}
.notes{{font-size:11px;color:var(--text-secondary);margin-top:12px;line-height:1.6;
        padding-top:12px;border-top:0.5px solid var(--border);}}
.notes strong{{color:var(--text-primary);font-weight:500;}}
.credit-bar{{display:flex;justify-content:flex-end;align-items:center;
             margin-top:14px;padding-top:12px;border-top:0.5px solid var(--border);}}
.d4tp-logo{{display:inline-block;text-decoration:none;opacity:0.85;transition:opacity 0.15s;}}
.d4tp-logo:hover{{opacity:1;}}
.d4tp-logo svg{{display:block;height:24px;width:auto;}}
.logo-light-mode{{display:inline-block;}} .logo-dark-mode{{display:none;}}
@media(prefers-color-scheme:dark){{
  .logo-light-mode{{display:none;}} .logo-dark-mode{{display:inline-block;}}
}}
@media(max-width:700px){{
  body{{padding:14px;}} .hdr h1{{font-size:18px;}}
  .ctrls{{grid-template-columns:1fr;}}
}}
</style>
</head>
<body>

<div class="hdr">
  <h1>U.S. Petroleum Inventories &#8212; Weekly Seasonality</h1>
  <p>Stocks by week of year (1&#8211;52), one line per year. {CURRENT_YEAR} bolded in red; prior years shaded by decade.</p>
</div>

<div class="ctrls">
  <div class="ctrl-grp">
    <label for="product-sel">Product</label>
    <select id="product-sel">
{select_options}
    </select>
  </div>
  <div class="ctrl-grp">
    <label>Decades</label>
    <div class="decade-pills" id="decade-pills">
      {decade_pills_html}
    </div>
  </div>
</div>

<div class="legend" id="legend">
{legend_items}
</div>

<div id="chart"></div>

<div class="notes">
  <strong>Source:</strong> <span id="source-note">{source_notes['crude_oil']}</span>
</div>

<div class="credit-bar">
  {credit_bar_html}
</div>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js" charset="utf-8"></script>
<script>
const CURRENT_YEAR  = {CURRENT_YEAR};
const CHART_DATA    = {json.dumps(chart_data)};
const SOURCE_NOTES  = {json.dumps(source_notes)};

// ── Decade palette ────────────────────────────────────────────────────────────
const DECADE_PALETTES = {{
  1980:{{lr:168,lg:168,lb:175,la:0.28,dr:85, dg:85, db:102,da:0.84}},
  1990:{{lr:120,lg:200,lb:135,la:0.28,dr:18, dg:122,db:42, da:0.86}},
  2000:{{lr:118,lg:168,lb:228,la:0.28,dr:10, dg:72, db:172,da:0.88}},
  2010:{{lr:248,lg:178,lb:82, la:0.28,dr:185,dg:85, db:5,  da:0.88}},
  2020:{{lr:200,lg:148,lb:228,la:0.28,dr:108,dg:35, db:162,da:0.88}},
}};

function decadeRgba(decade, frac) {{
  const p = DECADE_PALETTES[decade];
  const r = Math.round(p.lr + frac*(p.dr-p.lr));
  const g = Math.round(p.lg + frac*(p.dg-p.lg));
  const b = Math.round(p.lb + frac*(p.db-p.lb));
  const a = (p.la + frac*(p.da-p.la)).toFixed(2);
  return `rgba(${{r}},${{g}},${{b}},${{a}})`;
}}

// ── State ─────────────────────────────────────────────────────────────────────
let currentProduct = 'crude_oil';
const selectedDecades = new Set({json.dumps(all_decades)});

// ── Build traces ──────────────────────────────────────────────────────────────
function buildTraces(product) {{
  const series  = CHART_DATA[product];
  const prior   = series.filter(s => s.year < CURRENT_YEAR);

  // Intra-decade position — computed only for visible decades so frac stays smooth
  const decadeGroups = {{}};
  prior.forEach(s => {{
    if (selectedDecades.has(s.decade)) {{
      if (!decadeGroups[s.decade]) decadeGroups[s.decade] = [];
      decadeGroups[s.decade].push(s.year);
    }}
  }});

  const traces = [];

  prior.forEach(s => {{
    if (!selectedDecades.has(s.decade)) return;
    const grp  = decadeGroups[s.decade];
    const frac = grp.indexOf(s.year) / Math.max(grp.length - 1, 1);
    traces.push({{
      x: s.x, y: s.y, type: 'scatter', mode: 'lines',
      line: {{ color: decadeRgba(s.decade, frac), width: 1 }},
      hovertemplate: `Year: ${{s.year}}<br>Week: %{{x}}<br>%{{y:,.0f}} thousand barrels<extra></extra>`,
    }});
  }});

  // Current year always on top
  const cur = series.find(s => s.year === CURRENT_YEAR);
  if (cur) traces.push({{
    x: cur.x, y: cur.y, type: 'scatter', mode: 'lines',
    line: {{ color: '#c0392b', width: 2.8 }},
    hovertemplate: `Year: ${{CURRENT_YEAR}}<br>Week: %{{x}}<br>%{{y:,.0f}} thousand barrels<extra></extra>`,
  }});

  return traces;
}}

// ── Layout ────────────────────────────────────────────────────────────────────
const layout = {{
  margin: {{t:14, b:52, l:78, r:16}},
  plot_bgcolor:'rgba(0,0,0,0)', paper_bgcolor:'rgba(0,0,0,0)',
  xaxis:{{
    title:{{text:'Week of year',font:{{size:12}}}},
    tickmode:'array',
    tickvals:[1,3,5,7,9,11,13,15,17,19,21,23,25,27,29,31,33,35,37,39,41,43,45,47,49,51],
    range:[0.5,52.5], gridcolor:'rgba(210,215,225,0.7)', showgrid:true, zeroline:false,
  }},
  yaxis:{{
    title:{{text:'Thousand barrels',font:{{size:12}}}},
    tickformat:',', separatethousands:true,
    gridcolor:'rgba(210,215,225,0.7)', showgrid:true, zeroline:false,
  }},
  showlegend: false,
  font:{{family:'-apple-system,BlinkMacSystemFont,"Inter","Segoe UI",sans-serif',size:12}},
  height: 500,
}};
const config = {{responsive:true, displayModeBar:false}};

// ── Initial render ────────────────────────────────────────────────────────────
Plotly.newPlot('chart', buildTraces(currentProduct), layout, config);

// ── Sync legend opacity ───────────────────────────────────────────────────────
function syncLegend() {{
  document.querySelectorAll('#legend .legend-item[data-decade]').forEach(el => {{
    el.style.opacity = selectedDecades.has(+el.dataset.decade) ? '1' : '0.25';
  }});
}}

// ── Decade pill toggle ────────────────────────────────────────────────────────
document.querySelectorAll('.decade-pill').forEach(pill => {{
  const decade = +pill.dataset.decade;
  const color  = pill.dataset.color;

  function applyStyle() {{
    const on = selectedDecades.has(decade);
    pill.classList.toggle('off', !on);
    pill.style.background   = on ? color : '';
    pill.style.color        = on ? '#fff' : '';
    pill.style.borderColor  = on ? color : '';
  }}

  applyStyle();   // set initial state

  pill.addEventListener('click', () => {{
    if (selectedDecades.has(decade)) selectedDecades.delete(decade);
    else                              selectedDecades.add(decade);
    applyStyle();
    syncLegend();
    Plotly.react('chart', buildTraces(currentProduct), layout, config);
  }});
}});

syncLegend();

// ── Product switch ────────────────────────────────────────────────────────────
document.getElementById('product-sel').addEventListener('change', function() {{
  currentProduct = this.value;
  Plotly.react('chart', buildTraces(currentProduct), layout, config);
  document.getElementById('source-note').textContent = SOURCE_NOTES[currentProduct];
}});
</script>
</body>
</html>"""

with open(OUTPUT_V3, "w", encoding="utf-8") as f:
    f.write(HTML)
print(f"Saved: {OUTPUT_V3}  ({os.path.getsize(OUTPUT_V3):,} bytes)")

Saved: petroleum_seasonality_v3.html  (164,608 bytes)
